In [ ]:
import pandas as pd
import spacy
from nltk.corpus import stopwords
from collections import defaultdict

df = pd.read_table('df_taste_total.tsv', sep='\t', on_bad_lines='skip')

stop_words = set(stopwords.words('english'))
nlp = spacy.load('en_core_web_sm')

def lemmatize_word(word):
    doc = nlp(word)
    lemma = [token.lemma_ for token in doc][0]  # Prendi il primo lemma (spesso il più comune)
    return lemma

anni = df['year'].tolist()
taste_words = df['Taste_Word'].astype(str).tolist()

spans = ['1600-1699', '1700-1799', '1800-1899', '1900-1999']

word_span_year_dict = {span: defaultdict(lambda: defaultdict(int)) for span in spans}

for span in spans:
    myStart = int(span.split("-")[0])
    myEnd = int(span.split("-")[1])
    
    for y, words in zip(anni, taste_words):
        y = str(y).strip()
        y = y.replace(".0", "")
        
        if not y.isdigit():
            continue
        if len(y) < 4:
            continue
        
        y = int(y)
        if y >= myStart and y <= myEnd:
            for word in words.split('|'):
                for subword in word.strip().lower().split():  # Split by whitespace to get individual words
                    subword = subword.strip()
                    # Lemmatizza la parola utilizzando spaCy
                    lemma = lemmatize_word(subword)
                    if len(lemma) < 3 or lemma in stop_words:
                        continue
                    word_span_year_dict[span][y][lemma] += 1

percentage_dict = {span: defaultdict(dict) for span in spans}
for span in spans:
    for year in word_span_year_dict[span]:
        total_occurrences = sum(word_span_year_dict[span][year].values())
        for word in word_span_year_dict[span][year]:
            percentage_dict[span][year][word] = (word_span_year_dict[span][year][word] / total_occurrences) * 100

for span in sorted(percentage_dict.keys()):
    print(f"Span: {span}")
    for year in sorted(percentage_dict[span].keys()):
        print(f"  Year: {year}")
        for word in sorted(percentage_dict[span][year].keys()):
            print(f"    {word}: {percentage_dict[span][year][word]:.2f}%")

outfile = open('taste_lem_freq_span.tsv', 'w')

outfile.write("span\tyear\tword\tpercentage\n")

for span in sorted(percentage_dict.keys()):
    for year in sorted(percentage_dict[span].keys()):
        for word in sorted(percentage_dict[span][year].keys()):
            outfile.write(f"{span}\t{year}\t{word}\t{percentage_dict[span][year][word]:.2f}\n")

outfile.close()

In [ ]:
import csv

def read_data(file):
    data = defaultdict(lambda: defaultdict(float))
    
    with open(file, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter='\t')
        
        for row in reader:
            span = row['span']
            word = row['word']
            percentage = float(row['percentage'])
            data[span][word] += percentage
    
    return data


def calculate_relative_frequency_by_category(data, categories):
    category_frequency = defaultdict(lambda: defaultdict(float))
    
    for span, word_percentages in data.items():
        total_span = sum(word_percentages.values())
        
        for category, category_words in categories.items():
            category_frequency[category][span] = (
                sum(word_percentages[word] for word in category_words) / total_span
            )
    
    return category_frequency


def print_relative_frequency_by_category(category_frequency):
    for category, data in category_frequency.items():
        print(f"Relative frequency for the category '{category}':")
        
        for span, frequency in data.items():
            print(f"    Span {span}: {frequency:.2%}")
        
        print()


categories = {
    'savouriness': [
        'delicacy', 'savouriness', 'deliciousness', 'daintiness',
        'delicateness', 'toothsomeness', 'piquantness', 'palatableness',
        'fumet', 'fumette', 'tastefulness', 'palatability', 'smack',
        'relish', 'gust', 'hogo', 'zest', 'sapid', 'spice', 'spiciness',
        'tanginess', 'lickerous', 'delicious', 'delicate', 'dainty',
        'merry', 'dainteous', 'soft', 'daintiful', 'savourly', 'liking',
        'seasonable', 'licious', 'lusty', 'savorous', 'savourable',
        'daintive', 'savoury', 'exquisite', 'toothsome', 'well-relished',
        'taste-pleasing', 'daint', 'relishsome', 'savoursome', 'tastesome',
        'friand', 'relishing', 'lickerish', 'liquorish', 'well-relishing',
        'neat', 'palate-pleasing', 'tasteful', 'famelic', 'palate', 'tasty',
        'palatable', 'toothful', 'sipid', 'unsoured', 'tooth-tempting',
        'well-tasted', 'tastable', 'piquant', 'sapid', 'spicy', 'saporous',
        'slape', 'unctuous', 'palative', 'flavorous', 'well-flavoured',
        'gusty', 'flavoury', 'fine-palated', 'nutty', 'degustatory',
        'zesty', 'unrepulsive', 'peckish', 'mouth-watering', 'unreasty',
        'relishy', 'toothy', 'tasty-looking', 'flavoured', 'flavored',
        'flavory', 'hungrifying', 'unrancid', 'velvety', 'snappy',
        'tangy', 'delish', 'finger-licking', 'daintily', 'delicately',
        'dainteously', 'daintifully', 'savourily', 'savourously',
        'daintly', 'tastefully', 'palatably', 'toothsomely', 'sweet',
        'swete', 'sweetly', 'douce', 'soot', 'luscious', 'dulcet',
        'sugarish', 'honey', 'dulce', 'figgy', 'nectared', 'marmalady',
        'fat', 'unsharp', 'unsour', 'marmalade', 'ambrosian', 'dulcid',
        'dulcorous', 'dulceous', 'saccharaceous', 'saccharic', 'sugared',
        'sweetening', 'dulcific', 'dulcifying', 'sweetened', 'nectarized',
        'mellified', 'sugary', 'bitter-sweet', 'dulcified', 'edulcorate',
        'saccharous', 'saccharinized', 'mellifluous', 'melled',
        'sweetish', 'wallow-sweet', 'over-sweet', 'over-luscious',
        'sweeten'
    ],

    'unsavouriness': [
        'weffe', 'unsavouriness', 'nastiness', 'untoothsomeness', 'degout',
        'unpalatableness', 'unpalatability', 'soddenness', 'uneatableness',
        'rankness', 'restiness', 'rammishness', 'reasiness', 'rancidity',
        'rancidness', 'empyreuma', 'empyreumatism', 'hogo', 'haut-gout',
        'off-flavour', 'tinniness', 'unsavoury', 'unrelishable',
        'unrelishing', 'unsapory', 'insapory', 'unsweet', 'untasty',
        'untoothsome', 'twice', 'sod', 'coarse', 'irrelishable', 'aspre',
        'insuave', 'untoward', 'asperous', 'unpalatable', 'unsweetened',
        'impalatable', 'sodden', 'metallic', 'inky', 'weedy', 'tinny',
        'tangy', 'raw', 'unappetizing', 'twangy', 'stavy', 'toasty',
        'soapy', 'stewy', 'gloppy', 'uneatable', 'unedible', 'impotable',
        'undrinkable', 'unpotable', 'resty', 'rest', 'rammish', 'reezed',
        'musty', 'rusty', 'rank', 'turned', 'reasty', 'frowy', 'flatten',
        'rammy', 'reasy', 'rancid', 'loud', 'ranked', 'virous', 'ranciduous',
        'rafty', 'virose', 'loud-flavoured', 'empyreumatic',
        'empyreumatical', 'empyreumatized', 'foul', 'ful', 'queasy',
        'walsh', 'nasty', 'wallowish', 'fulsome', 'distastable',
        'distasteful', 'disgustful', 'nauseous', 'mawmish', 'mawkish',
        'disgusting', 'brackish', 'wambly', 'unsavourly', 'unsavourily',
        'undrinkably', 'rammishly', 'rancidly', 'unpalatably', 'sourly',
        'pungently', 'pungitively', 'eager', 'sourish', 'sour', 'sower',
        'crabbed', 'soured', 'souring', 'tart', 'acid', 'subacid',
        'acidolous', 'salso-acid', 'acescent', 'acetous', 'acidulent',
        'vinegarish', 'vinegary', 'acidy', 'acetic', 'bitter', 'sharp',
        'bask', 'gally', 'acrimonious', 'acrid', 'bitterish',
        'bitter-rinded'
    ],

    'insipidity': [
        'smatchless', 'fond', 'savourless', 'wershed', 'wearish', 'mild',
        'palled', 'dolled', 'unsavoured', 'walsh', 'wallowish', 'waterish',
        'flatten', 'seasonless', 'gustless', 'blown', 'flash', 'flat',
        'fatuous', 'tasteless', 'insipid', 'ingustable', 'flashy',
        'flatted', 'saltless', 'remiss', 'untasteable', 'vapid',
        'exolete', 'distasted', 'vappous', 'insulse', 'toothless',
        'mawkish', 'waugh', 'intastable', 'flavourless', 'impoignant',
        'instimulating', 'deadish', 'brineless', 'wishy-washy',
        'keestless', 'shilpit', 'sapidless', 'wish-washy', 'wersh',
        'silent', 'slushy', 'bland', 'spendsavour', 'spiceless',
        'untasting', 'palling', 'wearishness', 'tastelessness',
        'wallowishness', 'insipidity', 'flashiness', 'insipidness',
        'deadness', 'flatness', 'mawkishness', 'walshness',
        'ditchwateriness', 'savourlessness', 'blandness', 'wallowishly',
        'insipidly', 'mawkishly'
    ]

    # 'sweet': [
    #     'sweet', 'swete', 'sweetly', 'douce', 'soot', 'luscious',
    #     'dulcet', 'sugarish', 'honey', 'dulce', 'figgy', 'nectared',
    #     'marmalady', 'fat', 'unsharp', 'unsour', 'marmalade',
    #     'ambrosian', 'dulcid', 'dulcorous', 'dulceous', 'saccharaceous',
    #     'saccharic', 'sugared', 'sweetening', 'dulcific', 'dulcifying',
    #     'sweetened', 'nectarized', 'mellified', 'sugary',
    #     'bitter-sweet', 'dulcified', 'edulcorate', 'saccharous',
    #     'saccharinized', 'mellifluous', 'melled', 'sweetish',
    #     'wallow-sweet', 'over-sweet', 'over-luscious', 'sweeten'
    # ],

    # 'sour': [
    #     'sourly', 'pungently', 'pungitively', 'eager', 'sourish',
    #     'sour', 'sower', 'crabbed', 'soured', 'souring', 'tart'
    # ],

    # 'acid': [
    #     'acid', 'subacid', 'acidolous', 'salso-acid', 'acescent',
    #     'acetous', 'acidulent', 'vinegarish', 'vinegary', 'acidy',
    #     'acetic'
    # ],

    # 'salty': [
    #     'salty', 'salt', 'over-salt', 'fire-salt', 'saltish'
    # ],

    # 'bitter': [
    #     'bitter', 'sharp', 'bask', 'gally', 'acrimonious', 'acrid',
    #     'bitterish', 'bitter-rinded'
    # ]
}


data = read_data('taste_word_count_lemma.tsv')

relative_category_frequency = calculate_relative_frequency_by_category(
    data,
    categories
)

print_relative_frequency_by_category(relative_category_frequency)

Relative frequency for the category 'savouriness':
    Span 1600-1699: 27.39%
    Span 1700-1799: 10.80%
    Span 1800-1899: 19.88%
    Span 1900-1999: 20.06%

Relative frequency for the category 'unsavouriness':
    Span 1600-1699: 11.39%
    Span 1700-1799: 8.00%
    Span 1800-1899: 10.98%
    Span 1900-1999: 11.11%

Relative frequency for the category 'insipidity':
    Span 1600-1699: 1.14%
    Span 1700-1799: 2.12%
    Span 1800-1899: 1.87%
    Span 1900-1999: 1.14%



In [ ]:
def read_data(file):
    data = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    
    with open(file, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter='\t')
        
        for row in reader:
            span = row['span']
            year = row['year']
            word = row['word']
            percentage = float(row['percentage'])
            
            data[span][year][word] += percentage
    
    return data


def calculate_relative_frequency_by_category(data, categories):
    category_frequency = defaultdict(
        lambda: defaultdict(lambda: defaultdict(float))
    )
    
    for span, years in data.items():
        for year, word_percentages in years.items():
            total_year = sum(word_percentages.values())
            
            for category, category_words in categories.items():
                category_frequency[category][span][year] = (
                    sum(
                        word_percentages[word]
                        for word in category_words
                    ) / total_year
                )
    
    return category_frequency


def print_relative_frequency_by_category(category_frequency):
    for category, data in category_frequency.items():
        print(f"Relative frequency for the category '{category}':")
        
        for span, years in data.items():
            print(f"  Span {span}:")
            
            for year, frequency in sorted(years.items()):
                print(f"    Year {year}: {frequency:.2%}")
        
        print()



categories = {
    'savouriness': [
        'delicacy', 'savouriness', 'deliciousness', 'daintiness',
        'delicateness', 'toothsomeness', 'piquantness', 'palatableness',
        'fumet', 'fumette', 'tastefulness', 'palatability', 'smack',
        'relish', 'gust', 'hogo', 'zest', 'sapid', 'spice', 'spiciness',
        'tanginess', 'lickerous', 'delicious', 'delicate', 'dainty',
        'merry', 'dainteous', 'soft', 'daintiful', 'savourly', 'liking',
        'seasonable', 'licious', 'lusty', 'savorous', 'savourable',
        'daintive', 'savoury', 'exquisite', 'toothsome', 'well-relished',
        'taste-pleasing', 'daint', 'relishsome', 'savoursome',
        'tastesome', 'friand', 'relishing', 'lickerish', 'liquorish',
        'well-relishing', 'neat', 'palate-pleasing', 'tasteful',
        'famelic', 'palate', 'tasty', 'palatable', 'toothful', 'sipid',
        'unsoured', 'tooth-tempting', 'well-tasted', 'tastable',
        'piquant', 'sapid', 'spicy', 'saporous', 'slape', 'unctuous',
        'palative', 'flavorous', 'well-flavoured', 'gusty', 'flavoury',
        'fine-palated', 'nutty', 'degustatory', 'zesty', 'unrepulsive',
        'peckish', 'mouth-watering', 'unreasty', 'relishy', 'toothy',
        'tasty-looking', 'flavoured', 'flavored', 'flavory',
        'hungrifying', 'unrancid', 'velvety', 'snappy', 'tangy',
        'delish', 'finger-licking', 'daintily', 'delicately',
        'dainteously', 'daintifully', 'savourily', 'savourously',
        'daintly', 'tastefully', 'palatably', 'toothsomely'
    ],

    'unsavouriness': [
        'weffe', 'unsavouriness', 'nastiness', 'untoothsomeness',
        'degout', 'unpalatableness', 'unpalatability', 'soddenness',
        'uneatableness', 'rankness', 'restiness', 'rammishness',
        'reasiness', 'rancidity', 'rancidness', 'empyreuma',
        'empyreumatism', 'hogo', 'haut-gout', 'off-flavour',
        'tinniness', 'unsavoury', 'unrelishable', 'unrelishing',
        'unsapory', 'insapory', 'unsweet', 'untasty', 'untoothsome',
        'twice', 'sod', 'coarse', 'irrelishable', 'aspre', 'insuave',
        'untoward', 'asperous', 'unpalatable', 'unsweetened',
        'impalatable', 'sodden', 'metallic', 'inky', 'weedy', 'tinny',
        'tangy', 'raw', 'unappetizing', 'twangy', 'stavy', 'toasty',
        'soapy', 'stewy', 'gloppy', 'uneatable', 'unedible',
        'impotable', 'undrinkable', 'unpotable', 'resty', 'rest',
        'rammish', 'reezed', 'musty', 'rusty', 'rank', 'turned',
        'reasty', 'frowy', 'flatten', 'rammy', 'reasy', 'rancid',
        'loud', 'ranked', 'virous', 'ranciduous', 'rafty', 'virose',
        'loud-flavoured', 'empyreumatic', 'empyreumatical',
        'empyreumatized', 'foul', 'ful', 'queasy', 'walsh', 'nasty',
        'wallowish', 'fulsome', 'distastable', 'distasteful',
        'disgustful', 'nauseous', 'mawmish', 'mawkish', 'disgusting',
        'brackish', 'wambly', 'unsavourly', 'unsavourily',
        'undrinkably', 'rammishly', 'rancidly', 'unpalatably'
    ],

    'insipidity': [
        'smatchless', 'fond', 'savourless', 'wershed', 'wearish',
        'mild', 'palled', 'dolled', 'unsavoured', 'walsh',
        'wallowish', 'waterish', 'flatten', 'seasonless', 'gustless',
        'blown', 'flash', 'flat', 'fatuous', 'tasteless', 'insipid',
        'ingustable', 'flashy', 'flatted', 'saltless', 'remiss',
        'untasteable', 'vapid', 'exolete', 'distasted', 'vappous',
        'insulse', 'toothless', 'mawkish', 'waugh', 'intastable',
        'flavourless', 'impoignant', 'instimulating', 'deadish',
        'brineless', 'wishy-washy', 'keestless', 'shilpit',
        'sapidless', 'wish-washy', 'wersh', 'silent', 'slushy',
        'bland', 'spendsavour', 'spiceless', 'untasting', 'palling',
        'wearishness', 'tastelessness', 'wallowishness',
        'insipidity', 'flashiness', 'insipidness', 'deadness',
        'flatness', 'mawkishness', 'walshness', 'ditchwateriness',
        'savourlessness', 'blandness', 'wallowishly', 'insipidly',
        'mawkishly'
    ]
}


data = read_data('taste_lem_freq_span.tsv')

relative_category_frequency = calculate_relative_frequency_by_category(
    data,
    categories
)

print_relative_frequency_by_category(relative_category_frequency)

Relative frequency for the category 'savouriness':
  Span 1600-1699:
    Year 1604: 0.00%
    Year 1605: 23.87%
    Year 1613: 0.00%
    Year 1615: 0.00%
    Year 1616: 14.29%
    Year 1617: 0.00%
    Year 1620: 0.00%
    Year 1626: 4.76%
    Year 1629: 0.00%
    Year 1630: 0.00%
    Year 1637: 3.47%
    Year 1646: 0.00%
    Year 1649: 0.00%
    Year 1657: 3.57%
    Year 1661: 33.33%
    Year 1663: 0.00%
    Year 1665: 3.77%
    Year 1668: 0.00%
    Year 1669: 9.26%
    Year 1670: 1.44%
    Year 1674: 0.00%
    Year 1675: 0.00%
    Year 1678: 0.00%
    Year 1681: 8.39%
    Year 1682: 0.00%
    Year 1687: 8.70%
    Year 1693: 12.50%
    Year 1698: 4.55%
    Year 1699: 9.96%
  Span 1700-1799:
    Year 1700: 4.17%
    Year 1701: 0.00%
    Year 1702: 15.71%
    Year 1708: 0.00%
    Year 1709: 10.53%
    Year 1711: 6.90%
    Year 1715: 0.00%
    Year 1716: 9.68%
    Year 1718: 0.00%
    Year 1720: 0.00%
    Year 1721: 0.00%
    Year 1722: 0.00%
    Year 1723: 2.16%
    Year 1724: 0.00%
    

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_table(
    'span_word_percentage.tsv',
    on_bad_lines='skip'
)

data = {}

for _, row in df.iterrows():
    span = str(row['span'])
    word = row['word']
    freq = row['percentage']
    
    if span not in data:
        data[span] = {}
    
    data[span][word] = freq


def plot_top_words_per_category_normalized(data, categories, top_n=10):
    for category, category_words in categories.items():
        for century, words in data.items():
            
            category_words_century = {
                word: frequency
                for word, frequency in words.items()
                if word in category_words
            }
            
            total_frequencies = sum(category_words_century.values())
            
            if total_frequencies == 0:
                continue  # Skip if there are no frequencies for this category in this century
            
            sorted_words = sorted(
                category_words_century.items(),
                key=lambda x: x[1],
                reverse=True
            )[:top_n]


def print_top_words_per_category_normalized(data, categories, top_n=10):
    for category, category_words in categories.items():
        for century, words in data.items():
            
            category_words_century = {
                word: frequency
                for word, frequency in words.items()
                if word in category_words
            }
            
            total_frequencies = sum(category_words_century.values())
            
            sorted_words = sorted(
                category_words_century.items(),
                key=lambda x: x[1],
                reverse=True
            )[:top_n]

            top_words = [word[0] for word in sorted_words]
            top_frequencies = [
                word[1] / total_frequencies
                for word in sorted_words
            ]  # Normalize the frequency

            print(
                f'Top {top_n} most frequent words in the category '
                f'"{category}" for the Century {int(century)}:'
            )
            
            for word, frequency in zip(top_words, top_frequencies):
                print(f'{word}: {frequency:.2%}')
            
            print()


print_top_words_per_category_normalized(
    data,
    categories,
    top_n=10
)

Top 10 most frequent words in the category "savouriness" for the Century 1600:
dainty: 58.04%
delicious: 34.29%
relish: 7.67%

Top 10 most frequent words in the category "savouriness" for the Century 1700:
savoury: 41.71%
delicious: 17.11%
relish: 12.30%
dainty: 8.56%
palatable: 6.95%
spicy: 3.21%
gust: 1.60%
delicacy: 1.60%
smack: 1.60%
flavored: 1.07%

Top 10 most frequent words in the category "savouriness" for the Century 1800:
delicious: 38.85%
relish: 18.28%
palatable: 9.27%
delicacy: 8.73%
savoury: 7.91%
dainty: 4.34%
flavoured: 3.89%
sapid: 2.18%
spicy: 1.91%
flavored: 1.09%

Top 10 most frequent words in the category "savouriness" for the Century 1900:
delicious: 38.47%
dainty: 11.97%
palatable: 9.96%
savoury: 9.57%
relish: 7.57%
flavoured: 4.22%
tastefully: 3.26%
flavored: 3.25%
tasteful: 2.50%
delicacy: 2.26%

Top 10 most frequent words in the category "savouriness" for the Century 20:
delicious: 36.71%
dainty: 12.68%
savoury: 12.18%
flavoured: 8.16%
palatable: 8.05%
relish: